In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [116]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

In [117]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [118]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [119]:
categoricals = list(df.dtypes[df.dtypes == 'str'].index)
numericals = list(df.dtypes[df.dtypes != 'str'].index)
numericals.remove('converted')
categoricals, numericals

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'])

In [120]:
df[categoricals] = df[categoricals].fillna('NA')
df[numericals] = df[numericals].fillna(0)

Q1

In [121]:
df.industry.value_counts().sort_values(ascending=False)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Q2

In [122]:
print('interaction_count vs lead_score:', df[['interaction_count', 'lead_score']].corrwith(df['lead_score']))
print()
print('number_of_courses_viewed vs lead_score:', df[['number_of_courses_viewed', 'lead_score']].corrwith(df['lead_score']))
print()
print('number_of_courses_viewed vs interaction_count', df[['number_of_courses_viewed', 'interaction_count']].corrwith(df['interaction_count']))
print()
print('annual_income vs interaction_count', df[['annual_income', 'interaction_count']].corrwith(df['interaction_count']))

interaction_count vs lead_score: interaction_count    0.915746
lead_score           1.000000
dtype: float64

number_of_courses_viewed vs lead_score: number_of_courses_viewed    0.757204
lead_score                  1.000000
dtype: float64

number_of_courses_viewed vs interaction_count number_of_courses_viewed    0.721609
interaction_count           1.000000
dtype: float64

annual_income vs interaction_count annual_income        0.122842
interaction_count    1.000000
dtype: float64


splitting the data

In [123]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

Q3

In [124]:
from sklearn.metrics import mutual_info_score
print('MI with target:')
for c in categoricals:
    print(c, round(mutual_info_score(df_train['converted'], df_train[c]), 2))

MI with target:
lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


Q4

In [125]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)
train_dicts = df_train[categoricals + numericals].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
y_train = df_train['converted'].values

X_val = dv.transform(df_val[categoricals + numericals].to_dict(orient='records'))
y_val = df_val['converted'].values



model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)
accuracy = (churn_decision == y_val).mean()

round(accuracy, 2)



np.float64(0.64)

Q5

In [126]:
features = [f for f in (categoricals + numericals) if f != 'lead_source']
features

['industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [127]:
print(f"Accuracy with all features: {accuracy}")
print()
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    dv_selection = DictVectorizer(sparse=False)
    features = [c for c in categoricals + numericals if c!=f]

    train_dicts_selection = df_train[features].to_dict(orient='records')


    X_train_selection = dv_selection.fit_transform(train_dicts_selection)
    y_train_selection = df_train['converted'].values

    X_val_selection = dv_selection.transform(df_val[features].to_dict(orient='records'))
    y_val_selection = df_val['converted'].values



    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_selection, y_train_selection)

    y_pred = model.predict_proba(X_val_selection)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy_selection = (churn_decision == y_val_selection).mean()


    print(f"Accuracy without {f}: {accuracy_selection:.5f}")
    print(f"Accuracy difference: {abs(accuracy_selection - accuracy)}")
    print()



Accuracy with all features: 0.645

Accuracy without lead_source: 0.64200
Accuracy difference: 0.0030000000000000027

Accuracy without number_of_courses_viewed: 0.64300
Accuracy difference: 0.0020000000000000018

Accuracy without interaction_count: 0.60100
Accuracy difference: 0.04400000000000004



Q6

In [128]:
# c is regularization parameter, smaller values specify stronger regularization

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy_selection = (churn_decision == y_val).mean()


    print(f"Accuracy with C={c}: {accuracy_selection:.10f}")
    print()

Accuracy with C=1e-06: 0.5980000000

Accuracy with C=1e-05: 0.5980000000

Accuracy with C=0.0001: 0.6130000000

Accuracy with C=0.001: 0.6450000000

